# Uji Validitas dan Reliabilitas Instrumen Kuesioner
Data: respons kuesioner Google Form, hanya kelas 3 TI A (sesuai instruksi dosen).

## Langkah 1 – Persiapan lingkungan dan impor pustaka

In [ ]:
import re
import os
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 60)

# Konfigurasi
NAMA_FILE = "data/data_kuesioner.xlsx"   # file data yang dipakai
KELAS = "3 TI A"                         
FOLDER_OUTPUT = "output"
os.makedirs(f"{FOLDER_OUTPUT}/grafik", exist_ok=True)

## Langkah 2 – Membaca berkas data

In [3]:
print("File terbaca:", NAMA_FILE)
df_semua = pd.read_excel(NAMA_FILE)
print("Jumlah respons seluruh kelas:", len(df_semua))
print(df_semua["Kelas"].value_counts().to_string())

File terbaca: data/data_kuesioner.xlsx
Jumlah respons seluruh kelas: 181
Kelas
3 TI E    39
3 TI A    39
3 TI D    38
3 TI C    35
3 TI F    30


## Langkah 3 – Memfilter kelas dan memisahkan butir

In [ ]:
df = df_semua[df_semua["Kelas"].astype(str).str.strip() == KELAS].reset_index(drop=True)

# Butir = kolom berformat "Nama Dimensi [pernyataan]"
pola = r"^(.*?) \["
kolom_butir = [c for c in df.columns if re.match(pola, c)]
kolom_non_butir = [c for c in df.columns if c not in kolom_butir]
butir = df[kolom_butir]

print("Kelas               :", KELAS)
print("Jumlah responden (n):", len(df))
print("Jumlah butir        :", butir.shape[1])
print("Data kosong         :", int(butir.isna().sum().sum()))
print("Kolom non-butir     :", len(kolom_non_butir))
for c in kolom_non_butir:
    print("  -", c.strip())

Kelas               : 3 TI A
Jumlah responden (n): 39
Jumlah butir        : 29
Data kosong         : 0
Kolom non-butir     : 7
  - Timestamp
  - Score
  - Kelas
  - Jenis Kelamin
  - Jumlah Platform AI yang rutin digunakan untuk membatu menyelesaikan tugas atau projek
  - Jumlah Platform AI yang berlangganan secara premium (bukan free)
  - Durasi (jam) rata-rata penggunaan gadget perhari, untuk , kebutuhan harian sampai dengan hiburan


## Langkah 4 – Mengubah jawaban Likert menjadi angka

In [ ]:
skala = {
    "Sangat Tidak Setuju (STS)": 1,
    "Tidak Setuju (TS)": 2,
    "Netral (N)": 3,
    "Setuju (S)": 4,
    "Sangat Setuju (SS)": 5,
}
num = butir.apply(lambda s: s.astype(str).str.strip()).replace(skala).apply(pd.to_numeric, errors="coerce")
# Harus 0. Jika lebih dari 0, berarti ada jawaban yang tidak terbaca.
print("Sel yang gagal dikonversi:", int(num.isna().sum().sum()))
print(num.head().to_string(header=False))

Sel yang gagal dikonversi: 0
0  3  4  5  3  4  5  5  4  3  3  3  3  5  4  4  3  3  2  3  2  3  3  3  3  4  4  5  4  5
1  4  5  4  4  2  5  4  3  4  2  2  1  4  5  1  2  4  2  2  3  4  2  3  4  1  2  3  4  5
2  4  4  4  4  2  4  5  4  3  2  2  1  4  4  5  5  4  2  1  1  4  5  4  5  4  3  4  5  5
3  5  4  2  1  1  5  5  1  2  3  1  4  5  2  5  5  3  1  3  1  5  4  4  5  5  1  5  3  1
4  4  4  5  3  4  5  4  5  4  3  2  2  5  5  4  4  5  2  2  2  1  1  1  1  4  4  5  1  5


## Langkah 5 – Mengelompokkan butir per dimensi

In [6]:
INFO_DIMENSI = {
    # nama di formulir (sudah dirapikan) : (nama tampilan, kode, topik)
    "Penggunaan AI pada tugas atau projek kuliah": ("Penggunaan AI pada Tugas atau Projek Kuliah", "PA", 1),
    "Aspek Psikologis penggunaan AI pada tugas atau projek kuliah": ("Aspek Psikologis Penggunaan AI pada Tugas atau Projek Kuliah", "AP", 1),
    "Kebiasaan penggunaan teknologi digital": ("Kebiasaan Penggunaan Teknologi Digital", "KT", 2),
    "Prokrastinasi akademik": ("Prokrastinasi Akademik", "PR", 2),
    "Rsspon emosional": ("Respon Emosional", "RE", 2),
}
TOPIK = {
    1: "Penggunaan Platform AI di Kalangan Mahasiswa serta Dampaknya terhadap Aspek Psikologis/Perasaan saat Menyelesaikan Tugas",
    2: "Pengaruh Kebiasaan Penggunaan Teknologi Digital dan Prokrastinasi Akademik terhadap Tingkat Stres Akademik Mahasiswa",
}

dimensi = {}      # nama tampilan -> daftar kode butir
teks_butir = {}   # kode -> teks pernyataan
kolom_ke_kode = {}
for kolom in kolom_butir:
    nama_form = re.match(pola, kolom).group(1).strip().rstrip(".").strip()
    nama, kode, topik = INFO_DIMENSI[nama_form]
    daftar = dimensi.setdefault(nama, [])
    kode_butir = f"{kode}{len(daftar) + 1}"
    daftar.append(kode_butir)
    teks_butir[kode_butir] = re.search(r"\[(.*)\]\s*$", kolom).group(1).strip()
    kolom_ke_kode[kolom] = kode_butir

num = num.rename(columns=kolom_ke_kode)
topik_dimensi = {v[0]: v[2] for v in INFO_DIMENSI.values()}

for nama, daftar in dimensi.items():
    print(f"Topik {topik_dimensi[nama]} | {nama}: {len(daftar)} butir ({', '.join(daftar)})")

Topik 1 | Penggunaan AI pada Tugas atau Projek Kuliah: 7 butir (PA1, PA2, PA3, PA4, PA5, PA6, PA7)
Topik 1 | Aspek Psikologis Penggunaan AI pada Tugas atau Projek Kuliah: 7 butir (AP1, AP2, AP3, AP4, AP5, AP6, AP7)
Topik 2 | Kebiasaan Penggunaan Teknologi Digital: 6 butir (KT1, KT2, KT3, KT4, KT5, KT6)
Topik 2 | Prokrastinasi Akademik: 4 butir (PR1, PR2, PR3, PR4)
Topik 2 | Respon Emosional: 5 butir (RE1, RE2, RE3, RE4, RE5)


## Langkah 6 – Uji validitas (korelasi Pearson butir–total)

In [7]:
n = len(num)
t = stats.t.ppf(0.975, n - 2)
r_tabel = t / np.sqrt((n - 2) + t**2)
print(f"n = {n} | t kritis = {t:.4f} | r tabel (α=0,05; df={n-2}) = {r_tabel:.3f}")

def uji_validitas(data, dimensi):
    hasil = []
    for nama, daftar in dimensi.items():
        x = data[daftar]
        total = x.sum(axis=1)
        for i, kolom in enumerate(daftar, 1):
            r, p = stats.pearsonr(x[kolom], total)
            r_koreksi, _ = stats.pearsonr(x[kolom], total - x[kolom])
            hasil.append({
                "Dimensi": nama,
                "Butir": f"B{i}",
                "Kode": kolom,
                "r hitung": round(r, 3),
                "Corrected r": round(r_koreksi, 3),
                "p-value": round(p, 4),
                "Keterangan": "Valid" if r > r_tabel else "Tidak valid",
            })
    return pd.DataFrame(hasil)

tabel_validitas = uji_validitas(num, dimensi)
print(tabel_validitas.drop(columns="Dimensi").to_string(index=False))

n = 39 | t kritis = 2.0262 | r tabel (α=0,05; df=37) = 0.316
Butir Kode  r hitung  Corrected r  p-value  Keterangan
   B1  PA1     0.525        0.277   0.0006       Valid
   B2  PA2     0.269        0.105   0.0972 Tidak valid
   B3  PA3     0.643        0.379   0.0000       Valid
   B4  PA4     0.669        0.397   0.0000       Valid
   B5  PA5     0.515        0.126   0.0008       Valid
   B6  PA6     0.455        0.139   0.0036       Valid
   B7  PA7     0.016       -0.247   0.9237 Tidak valid
   B1  AP1     0.538        0.228   0.0004       Valid
   B2  AP2     0.645        0.377   0.0000       Valid
   B3  AP3     0.495        0.213   0.0014       Valid
   B4  AP4     0.558        0.326   0.0002       Valid
   B5  AP5     0.683        0.441   0.0000       Valid
   B6  AP6     0.129       -0.191   0.4329 Tidak valid
   B7  AP7     0.127       -0.146   0.4406 Tidak valid
   B1  KT1     0.584        0.268   0.0001       Valid
   B2  KT2     0.594        0.291   0.0001       Valid
   B

## Langkah 7 – Uji reliabilitas (Cronbach's Alpha) dan alpha jika butir dihapus

In [8]:
def cronbach_alpha(x):
    k = x.shape[1]
    return k / (k - 1) * (1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1))

def kategori(a):
    return ("Sangat baik" if a >= 0.90 else "Baik" if a >= 0.80
            else "Reliabel" if a >= 0.70 else "Tidak reliabel")

def uji_reliabilitas(data, dimensi):
    hasil_reliabilitas, alpha_dihapus = [], []
    for nama, daftar in dimensi.items():
        x = data[daftar]
        a = cronbach_alpha(x)
        hasil_reliabilitas.append({
            "Dimensi": nama,
            "Jumlah butir": len(daftar),
            "Cronbach's alpha": round(a, 3),
            "Kategori": kategori(a),
        })
        for i, kolom in enumerate(daftar, 1):
            if len(daftar) > 2:
                alpha_dihapus.append({
                    "Dimensi": nama,
                    "Butir": f"B{i}",
                    "Kode": kolom,
                    "Alpha jika dihapus": round(cronbach_alpha(x.drop(columns=kolom)), 3),
                })
    return pd.DataFrame(hasil_reliabilitas), pd.DataFrame(alpha_dihapus)

tabel_reliabilitas, tabel_alpha_dihapus = uji_reliabilitas(num, dimensi)
print(tabel_reliabilitas.to_string(index=False))
print()
print(tabel_alpha_dihapus.drop(columns="Dimensi").to_string(index=False))

                                                     Dimensi  Jumlah butir  Cronbach's alpha       Kategori
                 Penggunaan AI pada Tugas atau Projek Kuliah             7             0.370 Tidak reliabel
Aspek Psikologis Penggunaan AI pada Tugas atau Projek Kuliah             7             0.375 Tidak reliabel
                      Kebiasaan Penggunaan Teknologi Digital             6             0.478 Tidak reliabel
                                      Prokrastinasi Akademik             4             0.863           Baik
                                            Respon Emosional             5             0.489 Tidak reliabel

Butir Kode  Alpha jika dihapus
   B1  PA1               0.275
   B2  PA2               0.362
   B3  PA3               0.193
   B4  PA4               0.171
   B5  PA5               0.367
   B6  PA6               0.348
   B7  PA7               0.513
   B1  AP1               0.299
   B2  AP2               0.200
   B3  AP3               0.312
   B4  AP4

## Keputusan per butir
| Kondisi | Tindak lanjut |
|---|---|
| Valid dan alpha jika dihapus < α dimensi | Pertahankan |
| Tidak valid dan alpha jika dihapus > α dimensi | Buang atau revisi redaksi, lalu uji ulang |
| Valid, tetapi alpha jika dihapus > α dimensi | Tinjau isi butir |
| Tidak valid, tetapi alpha jika dihapus < α dimensi | Tinjau ulang sebelum dibuang |

In [9]:
def keputusan(row, alpha_dim):
    valid = row["Keterangan"] == "Valid"
    ad = row["Alpha jika dihapus"]
    if pd.isna(ad):
        return "Pertahankan" if valid else "Tinjau ulang"
    naik = ad > alpha_dim
    if valid and not naik:
        return "Pertahankan"
    if not valid and naik:
        return "Buang/revisi, uji ulang"
    if valid and naik:
        return "Tinjau isi butir"
    return "Tinjau ulang sebelum dibuang"

def gabungkan(tv, tad, tr):
    g = tv.merge(tad, on=["Dimensi", "Butir", "Kode"], how="left")
    alpha_map = tr.set_index("Dimensi")["Cronbach's alpha"]
    g["Alpha dimensi"] = g["Dimensi"].map(alpha_map)
    g["Keputusan"] = [keputusan(r, a) for (_, r), a in zip(g.iterrows(), g["Alpha dimensi"])]
    return g

gabung = gabungkan(tabel_validitas, tabel_alpha_dihapus, tabel_reliabilitas)
print(f"r tabel = {r_tabel:.3f}\n")
print(gabung[["Kode", "r hitung", "Corrected r", "Keterangan", "Alpha jika dihapus", "Alpha dimensi", "Keputusan"]].to_string(index=False))

r tabel = 0.316

Kode  r hitung  Corrected r  Keterangan  Alpha jika dihapus  Alpha dimensi                    Keputusan
 PA1     0.525        0.277       Valid               0.275          0.370                  Pertahankan
 PA2     0.269        0.105 Tidak valid               0.362          0.370 Tinjau ulang sebelum dibuang
 PA3     0.643        0.379       Valid               0.193          0.370                  Pertahankan
 PA4     0.669        0.397       Valid               0.171          0.370                  Pertahankan
 PA5     0.515        0.126       Valid               0.367          0.370                  Pertahankan
 PA6     0.455        0.139       Valid               0.348          0.370                  Pertahankan
 PA7     0.016       -0.247 Tidak valid               0.513          0.370      Buang/revisi, uji ulang
 AP1     0.538        0.228       Valid               0.299          0.375                  Pertahankan
 AP2     0.645        0.377       Valid        

## Uji ulang (iterasi Langkah 5–7)

In [10]:
riwayat = []
akhir = {}
for nama, daftar_awal in dimensi.items():
    daftar = list(daftar_awal)
    putaran = 0
    while True:
        sub = {nama: daftar}
        tv = uji_validitas(num, sub)
        tr, tad = uji_reliabilitas(num, sub)
        g = gabungkan(tv, tad, tr)
        a = tr["Cronbach's alpha"].iloc[0]
        kandidat = g[g["Keputusan"] == "Buang/revisi, uji ulang"].sort_values("r hitung")
        riwayat.append({
            "Dimensi": nama,
            "Putaran": putaran,
            "Butir dipakai": ", ".join(daftar),
            "Jumlah butir": len(daftar),
            "Cronbach's alpha": a,
            "Kategori": kategori(a),
            "Butir tidak valid": ", ".join(g.loc[g["Keterangan"] == "Tidak valid", "Kode"]) or "-",
            "Dibuang berikutnya": kandidat["Kode"].iloc[0] if len(kandidat) else "-",
        })
        if len(kandidat) == 0 or len(daftar) <= 2:
            akhir[nama] = g
            break
        daftar.remove(kandidat["Kode"].iloc[0])
        putaran += 1

tabel_iterasi = pd.DataFrame(riwayat)
print(tabel_iterasi.drop(columns="Dimensi").to_string(index=False))
print()
for nama, g in akhir.items():
    print(f"== Akhir: {nama}")
    print(g[["Kode", "r hitung", "Corrected r", "Keterangan", "Alpha jika dihapus", "Alpha dimensi", "Keputusan"]].to_string(index=False))

 Putaran                     Butir dipakai  Jumlah butir  Cronbach's alpha       Kategori Butir tidak valid Dibuang berikutnya
       0 PA1, PA2, PA3, PA4, PA5, PA6, PA7             7             0.370 Tidak reliabel          PA2, PA7                PA7
       1      PA1, PA2, PA3, PA4, PA5, PA6             6             0.513 Tidak reliabel               PA2                PA2
       2           PA1, PA3, PA4, PA5, PA6             5             0.530 Tidak reliabel                 -                  -
       0 AP1, AP2, AP3, AP4, AP5, AP6, AP7             7             0.375 Tidak reliabel          AP6, AP7                AP7
       1      AP1, AP2, AP3, AP4, AP5, AP6             6             0.479 Tidak reliabel               AP6                AP6
       2           AP1, AP2, AP3, AP4, AP5             5             0.668 Tidak reliabel                 -                  -
       0      KT1, KT2, KT3, KT4, KT5, KT6             6             0.478 Tidak reliabel               KT5    

## Statistik deskriptif per butir dan per dimensi

In [11]:
def ringkas(s):
    return {
        "n": int(s.count()),
        "Rata-rata": round(s.mean(), 3),
        "Median": s.median(),
        "Modus": s.mode().iloc[0],
        "Simpangan baku": round(s.std(ddof=1), 3),
        "Min": s.min(),
        "Maks": s.max(),
        "Skewness": round(s.skew(), 3),
        "Kurtosis": round(s.kurt(), 3),
    }

label = ["STS", "TS", "N", "S", "SS"]
baris_butir, baris_dimensi, komposisi = [], [], []
for nama, daftar in dimensi.items():
    for kode in daftar:
        baris_butir.append({"Dimensi": nama, "Kode": kode, **ringkas(num[kode])})
        frek = num[kode].value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)
        komposisi.append({"Dimensi": nama, "Kode": kode,
                          **{l: int(f) for l, f in zip(label, frek)},
                          **{f"%{l}": round(f / n * 100, 1) for l, f in zip(label, frek)}})
    total = num[daftar].sum(axis=1)
    baris_dimensi.append({"Dimensi": nama, "Jumlah butir": len(daftar),
                          "Rentang teoretis": f"{len(daftar)}–{5*len(daftar)}", **ringkas(total)})

desk_butir = pd.DataFrame(baris_butir)
desk_dimensi = pd.DataFrame(baris_dimensi)
tabel_komposisi = pd.DataFrame(komposisi)
print(desk_dimensi.to_string(index=False))
print()
print(desk_butir.drop(columns="Dimensi").to_string(index=False))
print()
print(tabel_komposisi.drop(columns="Dimensi").to_string(index=False))

                                                     Dimensi  Jumlah butir Rentang teoretis  n  Rata-rata  Median  Modus  Simpangan baku  Min  Maks  Skewness  Kurtosis
                 Penggunaan AI pada Tugas atau Projek Kuliah             7             7–35 39     27.282    27.0     27           3.017   19    34    -0.425     1.612
Aspek Psikologis Penggunaan AI pada Tugas atau Projek Kuliah             7             7–35 39     21.923    21.0     21           3.737   15    30     0.248    -0.752
                      Kebiasaan Penggunaan Teknologi Digital             6             6–30 39     16.692    17.0     18           3.722    8    24    -0.255    -0.212
                                      Prokrastinasi Akademik             4             4–20 39     11.051    11.0      8           4.065    4    19     0.236    -0.553
                                            Respon Emosional             5             5–25 39     19.385    20.0     21           3.322    7    25    -1.524   

## Grafik komposisi jawaban per dimensi

In [12]:
WARNA = ["#b8312f", "#ec9a99", "#d6d5d1", "#8db8ea", "#2a6fc4"]   # STS → SS (merah–abu–biru)
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9})

def grafik_komposisi(nama, daftar, nama_file):
    data = tabel_komposisi.set_index("Kode").loc[daftar, [f"%{l}" for l in label]]
    fig, ax = plt.subplots(figsize=(7.2, 0.42 * len(daftar) + 1.3), dpi=200)
    kiri = np.zeros(len(daftar))
    y = np.arange(len(daftar))[::-1]
    for j, l in enumerate(label):
        nilai = data[f"%{l}"].values
        ax.barh(y, nilai, left=kiri, color=WARNA[j], edgecolor="white", linewidth=1.2, height=0.7, label=l)
        for yi, k, v in zip(y, kiri, nilai):
            if v >= 5:
                ax.text(k + v / 2, yi, f"{v:.0f}", ha="center", va="center", fontsize=7.5,
                        color="white" if j in (0, 4) else "#222222")
        kiri += nilai
    ax.set_yticks(y)
    ax.set_yticklabels(daftar)
    ax.set_xlim(0, 100)
    ax.set_xlabel("Persentase responden (%)")
    for s in ["top", "right", "left"]:
        ax.spines[s].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.legend(ncol=5, loc="lower center", bbox_to_anchor=(0.5, 1.0), frameon=False, fontsize=8)
    ax.set_title(f"{nama} (n = {n})", fontsize=9.5, pad=24, loc="left")
    fig.tight_layout()
    fig.savefig(nama_file, bbox_inches="tight")
    plt.close(fig)
    return nama_file

for nama, daftar in dimensi.items():
    kode = daftar[0][:2]
    print("Disimpan:", grafik_komposisi(nama, daftar, f"{FOLDER_OUTPUT}/grafik/komposisi_{kode}.png"))

Disimpan: output/grafik/komposisi_PA.png
Disimpan: output/grafik/komposisi_AP.png
Disimpan: output/grafik/komposisi_KT.png
Disimpan: output/grafik/komposisi_PR.png
Disimpan: output/grafik/komposisi_RE.png


## Langkah 8 – Menggabungkan dan menyimpan hasil

In [13]:
daftar_butir = pd.DataFrame([{"Kode": k, "Dimensi": d, "Butir soal": teks_butir[k]}
                             for d, ks in dimensi.items() for k in ks])
with pd.ExcelWriter(f"{FOLDER_OUTPUT}/hasil_uji_validitas_reliabilitas.xlsx") as w:
    gabung.to_excel(w, sheet_name="Validitas", index=False)
    tabel_reliabilitas.to_excel(w, sheet_name="Reliabilitas", index=False)
    tabel_iterasi.to_excel(w, sheet_name="Uji ulang", index=False)
    desk_dimensi.to_excel(w, sheet_name="Deskriptif dimensi", index=False)
    desk_butir.to_excel(w, sheet_name="Deskriptif butir", index=False)
    tabel_komposisi.to_excel(w, sheet_name="Komposisi jawaban", index=False)
    daftar_butir.to_excel(w, sheet_name="Daftar butir", index=False)
    pd.DataFrame({"Info": ["Kelas", "n", "r tabel", "t kritis", "df"],
                  "Nilai": [KELAS, n, round(r_tabel, 4), round(t, 4), n - 2]}).to_excel(w, sheet_name="Info", index=False)
print("Disimpan:", f"{FOLDER_OUTPUT}/hasil_uji_validitas_reliabilitas.xlsx")

Disimpan: output/hasil_uji_validitas_reliabilitas.xlsx
